# SFT — Instruction tuning multi-turn (Section 6.2)

**Chỉ chạy sau khi Bước 1 (CPT) đã xong** — tức repo `{HF_USER}/Qwen3-1.7B-vi-cpt` (bản merged) đã tồn tại trên Hub. Notebook này dạy model CPT format hội thoại trợ lý trên `5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca` (12.7k hội thoại multi-turn tiếng Việt).

Cùng cơ chế với `CPT-Step-B-Train-Qwen3-1.7B.ipynb`: **Run All mỗi session** — tự kéo checkpoint mới nhất từ Hub → train tiếp → push checkpoint đầy đủ mỗi `save_steps` → tự dừng trước khi hết giờ. Ước tính ~800 step (2 epoch, effective batch 32) ≈ **1 session T4 là xong**, nhưng resume vẫn có sẵn phòng session bị cắt.

Repo trên Hub:
- `{HF_USER}/Qwen3-1.7B-vi-cpt` — input (output Bước 1, đã merge)
- `{HF_USER}/Qwen3-1.7B-vi-sft-ckpt` — checkpoint để resume
- `{HF_USER}/Qwen3-1.7B-vi-sft` — bản merged cuối, input cho Bước 3 (Reward Model)

**Trước khi chạy:** bật GPU T4 x1; secret `HF_TOKEN` (**Write** token — cell Config tự kiểm tra). Username HF tự lấy từ token, không cần sửa gì trong notebook.

In [ ]:
!pip install -q -U unsloth

In [ ]:
# Config + login — username TỰ LẤY từ token (whoami), không gõ tay để khỏi sai namespace
import os, time
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # chống phân mảnh VRAM — phải đặt TRƯỚC khi import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download, whoami
login(UserSecretsClient().get_secret("HF_TOKEN"))

info = whoami()
HF_USER = info["name"]
role = ((info.get("auth") or {}).get("accessToken") or {}).get("role")
assert role != "read", "HF_TOKEN là READ token — tạo token WRITE tại hf.co/settings/tokens rồi cập nhật Kaggle Secret"
print(f"HF user: {HF_USER} (token: {role})")

# W&B (tùy chọn): thêm secret WANDB_API_KEY (lấy tại wandb.ai/authorize) là tự bật dashboard;
# không có secret thì train vẫn chạy bình thường, log nằm trong trainer_state.json của checkpoint
try:
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_PROJECT"] = "vi-chatbot-rlhf"
    REPORT_TO = "wandb"
    print("W&B: BẬT — xem dashboard tại wandb.ai, project vi-chatbot-rlhf")
except Exception:
    REPORT_TO = "none"
    print("*** W&B: TẮT — secret WANDB_API_KEY không đọc được. Nếu bạn ĐÃ tạo secret mà vẫn thấy "
          "dòng này: secret chưa được TICK cho notebook này (Add-ons → Secrets → tick checkbox). "
          "Train vẫn chạy bình thường, log loss vẫn in ra Logs bên dưới. ***")

CPT_MODEL  = f"{HF_USER}/Qwen3-1.7B-vi-cpt"       # output Bước 1 (đã merge) — KHÔNG phải Qwen3 gốc
CKPT_REPO  = f"{HF_USER}/Qwen3-1.7B-vi-sft-ckpt"  # checkpoint resume
FINAL_REPO = f"{HF_USER}/Qwen3-1.7B-vi-sft"       # bản merged cuối — input cho Bước 3 (RM)
OUT_DIR    = "/kaggle/working/sft-checkpoints"
MAX_SEQ    = 2048
EPOCHS     = 2       # 12.7k hội thoại, effective batch 32 → ~800 step
BUDGET_H   = 10.0    # tính từ lúc load model; session batch sống ~11.5-12h (đo thực tế ở bước CPT) → còn dư an toàn
SYSTEM_PROMPT = "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."

In [ ]:
# Kéo checkpoint mới nhất từ Hub về (None nếu là session đầu tiên)
api = HfApi()
api.create_repo(CKPT_REPO, private=True, exist_ok=True)
steps = sorted(int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
               if f.startswith("checkpoint-"))
last_ckpt = None
if steps:
    name = f"checkpoint-{steps[-1]}"
    snapshot_download(CKPT_REPO, allow_patterns=f"{name}/*", local_dir=OUT_DIR)
    last_ckpt = os.path.join(OUT_DIR, name)
print("Resume từ:", last_ckpt or "(bắt đầu mới)")

In [ ]:
# Load model CPT đã merge (4-bit) + gắn adapter QLoRA r=16 — cấu hình cố định của Bước 2 (bảng 6.2)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    CPT_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","up_proj","down_proj","o_proj","gate_proj"],
    use_rslora=True, use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
# Data: 5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca — dạng ShareGPT ({"from": human/gpt, "value": ...})
# standardize_sharegpt → {"role", "content"}, rồi render thành ChatML với system prompt cố định.
# Dùng template "chatml" thuần thay template Qwen3 gốc (template gốc chèn block <think> — không cần
# cho SFT thường; token <|im_start|>/<|im_end|> vốn có sẵn trong vocab Qwen nên không phải thêm token)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

ds = standardize_sharegpt(load_dataset("5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca", split="train"))
def to_text(ex):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + ex["conversations"]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
ds = ds.map(to_text, remove_columns=ds.column_names).train_test_split(test_size=200, seed=42)
print(ds)
print(ds["train"][0]["text"][:600])

In [ ]:
# Callback: mỗi lần save, upload NGUYÊN thư mục checkpoint lên Hub (cả optimizer/scheduler/rng —
# session sau mới resume đúng). Giống hệt CPT-Step-B-Train-Qwen3-1.7B.ipynb:
#  - save mỗi 20 step → on_save retry 3 lần khi push lỗi (push hỏng KHÔNG được giết train)
#    và dọn checkpoint cũ trên Hub (giữ 2 bản + squash history) để repo ckpt không phình
#  - ngân sách giờ kiểm tra ở on_step_end: đến giờ là ÉP save ngay tại step hiện tại rồi dừng
# Hai lớp log, đều in TEXT ra stdout (bảng HTML widget của transformers KHÔNG hiện trong tab Logs
# khi chạy batch — Save & Run All — nên nhìn như treo dù đang train):
#  - on_step_end: [heartbeat] mỗi 20 step — độc lập với trainer.log, chắc chắn nổ nếu loop còn chạy,
#    ghi thẳng perf/sec_per_step lên W&B (không qua WandbCallback)
#  - on_log: loss/lr thật — nếu thấy heartbeat mà KHÔNG thấy dòng loss = trainer không bắn on_log
#    (bug Unsloth × transformers 5.x) → báo lại để vá tiếp
from transformers import TrainerCallback
class KaggleSessionCallback(TrainerCallback):
    def __init__(self):
        self.t0 = time.time(); self._last = (0, self.t0)
    def on_train_begin(self, args, state, control, **kw):
        self._last = (state.global_step, time.time())  # khi resume, global_step > 0 — không reset thì heartbeat đầu tính s/step sai
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 1 or state.global_step % 20 == 0:
            now = time.time()
            spd = (now - self._last[1]) / max(state.global_step - self._last[0], 1)
            self._last = (state.global_step, now)
            print(f"[heartbeat] step {state.global_step}/{state.max_steps}, {spd:.0f} s/step, "
                  f"{(now - self.t0)/3600:.2f}h", flush=True)
            if REPORT_TO == "wandb":
                import wandb
                if wandb.run: wandb.log({"perf/sec_per_step": spd}, step=state.global_step)
        if time.time() - self.t0 > BUDGET_H * 3600 and not control.should_training_stop:
            control.should_save = True
            control.should_training_stop = True
            print(f"Hết ngân sách {BUDGET_H}h — save ngay tại step {state.global_step} rồi dừng, "
                  "session sau tự resume.", flush=True)
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            msg = ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in logs.items())
            print(f"[step {state.global_step}/{state.max_steps} | {(time.time()-self.t0)/3600:.2f}h] {msg}", flush=True)
    def on_save(self, args, state, control, **kw):
        name = f"checkpoint-{state.global_step}"
        for attempt in range(3):
            try:
                api.upload_folder(repo_id=CKPT_REPO, folder_path=os.path.join(args.output_dir, name),
                                  path_in_repo=name)
                print(f"Đã push {name} lên {CKPT_REPO}", flush=True)
                break
            except Exception as e:
                print(f"Push {name} lỗi lần {attempt+1}/3 ({type(e).__name__}: {e}) — thử lại sau 30s",
                      flush=True)
                time.sleep(30)
        try:  # dọn Hub: giữ 2 checkpoint mới nhất; lỗi thì bỏ qua — dọn dẹp không được phép giết train
            steps = sorted({int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
                            if f.startswith("checkpoint-")})
            for s in steps[:-2]:
                api.delete_folder(path_in_repo=f"checkpoint-{s}", repo_id=CKPT_REPO)
            if steps[:-2]:
                api.super_squash_history(CKPT_REPO)  # xóa folder chưa giải phóng LFS — phải squash history
                print(f"Đã dọn Hub, còn checkpoint: {steps[-2:]}", flush=True)
        except Exception as e:
            print(f"(bỏ qua) dọn checkpoint cũ trên Hub lỗi: {type(e).__name__}: {e}", flush=True)

In [ ]:
# SFTTrainer + train_on_responses_only: chỉ tính loss trên lượt assistant (mask câu hỏi của user)
# Nếu TRL bản mới báo lỗi tham số: chuyển dataset_text_field/max_seq_length sang trl.SFTConfig
# fp16/bf16 tự chọn theo GPU: T4 (Turing) KHÔNG có bf16 → phải fp16; A100/L4 thì bf16
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# W&B: TỰ init trước, timeout 300s — KHÔNG để WandbCallback tự init (mặc định timeout 90s,
# không retry, CommError ném trong on_train_begin làm CHẾT CẢ TRAIN — đã dính ở bước CPT).
# W&B lỗi thì hạ xuống "none" và train tiếp.
RUN_NAME = f"sft-{time.strftime('%m%d-%H%M')}"
if REPORT_TO == "wandb":
    try:
        import wandb
        wandb.init(project=os.environ["WANDB_PROJECT"], name=RUN_NAME,
                   settings=wandb.Settings(init_timeout=300))
        print(f"W&B run: {wandb.run.url}", flush=True)
    except Exception as e:
        REPORT_TO = "none"
        print(f"*** W&B: init lỗi ({type(e).__name__}: {e}) — TẮT W&B, train vẫn chạy, "
              "log loss vẫn in ra Logs bên dưới. ***", flush=True)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    callbacks=[KaggleSessionCallback()],
    args=TrainingArguments(
        output_dir=OUT_DIR,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,  # effective 32 như script gốc
        learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=40,  # ≈ 5% của ~800 step (warmup_ratio deprecated từ transformers 5.x)
        num_train_epochs=EPOCHS,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        optim="paged_adamw_8bit",
        average_tokens_across_devices=False,  # transformers 5.x mặc định True → vỡ fused loss của Unsloth ('int' has no .mean, issue #3695)
        # save mỗi 20 step như CPT — session chết bất ngờ chỉ mất ≤20 step; on_save tự retry push
        # + dọn checkpoint cũ trên Hub (xem cell callback)
        save_strategy="steps", save_steps=20, save_total_limit=2,
        # Eval OOM trên T4 (dính ở bước CPT): lúc eval model trả nguyên logits [batch, 2048, vocab 151k]
        # rồi accelerate ép fp32 — batch eval mặc định cần ~18.5 GiB. batch 2 → ~2.5 GiB, an toàn.
        # eval mỗi 200 step, mốc chẵn: save-20 dày rồi nên eval có chết thì bản save-180 đã trên Hub.
        per_device_eval_batch_size=2, prediction_loss_only=True,
        eval_strategy="steps", eval_steps=200,
        logging_steps=10, logging_first_step=True,  # log ngay step 1 — chạy 2-3 phút là biết sống hay chết
        report_to=REPORT_TO, run_name=RUN_NAME,  # WandbCallback thấy wandb.run có sẵn thì dùng lại, không init nữa
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train(resume_from_checkpoint=last_ckpt)
print(f"global_step = {trainer.state.global_step} / {trainer.state.max_steps}")

In [ ]:
# Khi train đủ số step: merge LoRA vào model CPT, push bản final — Bước 3 (RM) load thẳng repo này
if trainer.state.global_step >= trainer.state.max_steps:
    model.push_to_hub_merged(FINAL_REPO, tokenizer, save_method="merged_16bit")
    print(f"SFT HOÀN TẤT — bản merged tại {FINAL_REPO}. Cập nhật README (bước 2 → done).")
else:
    print("Chưa xong — mở lại notebook này ở session sau, nó tự resume.")

In [ ]:
# Smoke test: model giờ phải trả lời theo format trợ lý (so tiêu chí 6.9 trong spec)
FastLanguageModel.for_inference(model)
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Xin chào! Bạn có thể giới thiệu về Việt Nam trong 3 câu không?"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=200, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))